# FastMCP包快速开发
## FastMCP的核心概念
### 服务器
FastMCP服务器是您与MCP协议的核心接口，处理连接管理、协议合规和消息路由：

```bash
mcp = FastMCP("My App", dependencies=["pandas", "numpy"])

```
### 资源
资源用于向LLMs公开数据，类似于REST API中的GET端点。例如：
```python
@mcp.resource("config://app")
def get_config() -> str:
    """静态配置数据"""
    return "App configuration here"
```

动态资源示例：

```python
@mcp.resource("users://{user_id}/profile")
def get_user_profile(user_id: str) -> str:
    """动态用户数据"""
    return f"Profile data for user {user_id}"
```
### 工具
工具使LLMs通过您的服务器执行操作，类似于POST端点。例如：
```python
@mcp.tool()
def calculate_bmi(weight_kg: float, height_m: float) -> float:
    """计算给定体重和身高的BMI"""
    return weight_kg / (height_m ** 2)
```

### 提示
提示是帮助LLMs有效地与您的服务器交互的可重用模板：
```python
@mcp.prompt()
def review_code(code: str) -> str:
    return f"请审核这段代码:\n\n{code}"
```


## 运行您的服务器
### 开发模式（推荐用于构建和测试)
```python
fastmcp dev server.py
```
这种模式提供了一个Web界面，可以测试工具和资源，查看详细日志等。

### Claude桌面集成（用于常规使用）
服务器声明后，可以通过以下命令安装在Claude Desktop中：

```python
fastmcp install server.py
```
### 直接执行（用于高级用例）
您还可以通过以下方式直接执行您的服务器：

```python
mcp.run()
```




## 示例
### Echo服务器
```python

from fastmcp import FastMCP

mcp = FastMCP("Echo")

@mcp.resource("echo://{message}")
def echo_resource(message: str) -> str:
    """回显消息作为资源"""
    return f"Resource echo: {message}"

@mcp.tool()
def echo_tool(message: str) -> str:
    """回显消息作为工具"""
    return f"Tool echo: {message}"

```

### SQLite Explorer
```python

from fastmcp import FastMCP
import sqlite3

mcp = FastMCP("SQLite Explorer")

@mcp.resource("schema://main")
def get_schema() -> str:
    """提供数据库模式作为资源"""
    conn = sqlite3.connect("database.db")
    schema = conn.execute("SELECT sql FROM sqlite_master WHERE type='table'").fetchall()
    return "\n".join(sql[0] for sql in schema if sql[0])

@mcp.tool()
def query_data(sql: str) -> str:
    """安全执行SQL查询"""
    conn = sqlite3.connect("database.db")
    try:
        result = conn.execute(sql).fetchall()
        return "\n".join(str(row) for row in result)
    except Exception as e:
        return f"Error: {str(e)}"
```
